# 03 · The held-out women and the development folds

Every number the study reports for a model comes from women the model never saw. A fifth of
the survey's **clusters** (the villages and neighbourhoods the survey visited) are set aside
before anything is fitted: 1,375 women. The rest, 5,518 women, are the **development** set,
split again into five folds for tuning.

Why clusters and not women: women in one cluster share a health post, a road and a season of
fieldwork. Splitting them between development and held-out would let a model learn the
cluster and look better than it is. The folds keep clusters whole for the same reason, and
keep each survey stratum (region by urban/rural) spread across the folds.

The split is drawn once, from a fixed seed, and every model at every rung uses exactly these
women and these folds.

In [ ]:
# Settings for this notebook
QUICK = False   # True: fewer resamples for the intervals (a check that everything runs)

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib

from anc8 import cohorts, design
dev, held = lib.partitions()   # also checks the split and folds against the study's digests
folds = design.make_psu_stratum_folds(dev)
print(f"development: {len(dev):,} women in {dev.v021.nunique()} clusters")
print(f"held-out:    {len(held):,} women in {held.v021.nunique()} clusters")
repro.check("no cluster is on both sides", not (set(dev.v021) & set(held.v021)))

In [ ]:
w = dev.v005 / 1e6
table = pd.DataFrame({
    "women": folds.value_counts().sort_index(),
    "clusters": dev.groupby(folds.to_numpy()).v021.nunique(),
    **{f"{r} (weighted)": dev.assign(f=folds.to_numpy(), w=w).groupby("f").apply(lambda g, r=r: np.average(g[r], weights=g.w))
       for r in lib.RUNGS},
})
table.index.name = "fold"
print(table.round(3).to_string())
print("\nEach fold has a similar share reaching each rung, so no fold is easier than another.")
repro.check("the split and folds are the study's (digests)", True, "checked when the partition was loaded")
repro.report()